### **HW2**
202315163 컴퓨터과학과 김소희

**1. Trigram Language Model을 구현하고, 이를 이용하여 5개의 sentence를 프린트하세요.**

In [1]:
from collections import Counter
import random
import math


# =========================================================
# 1. Training Corpus
# =========================================================

sentences = [
    "i want chinese food",
    "i want thai food",
    "i want italian food",
    "i want a restaurant",
    "i want to eat",
    "i want to eat chinese food",
    "i want to eat thai food",
    "i am looking for chinese food",
    "i am looking for a restaurant",
    "can you find a chinese restaurant",
    "can you find a thai restaurant",
    "can you find an italian restaurant",
    "tell me about chinese restaurants",
    "tell me about thai restaurants",
    "where can i eat chinese food",
]


# =========================================================
# 2. Tokenization
# =========================================================

def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]


corpus = [tokenize(s) for s in sentences]


# =========================================================
# 3. Unigram Counts
# =========================================================

unigram_counts = Counter()

for sentence in corpus:
    unigram_counts.update(sentence)

total_words = sum(unigram_counts.values())


def unigram_prob(word):
    return unigram_counts[word] / total_words


# =========================================================
# 4. Bigram Counts
# =========================================================

bigram_counts = Counter()

for sentence in corpus:
    for i in range(len(sentence) - 1):
        bigram = (sentence[i], sentence[i + 1])
        bigram_counts[bigram] += 1


def bigram_prob(previous_word, word):
    numerator = bigram_counts[(previous_word, word)]
    denominator = unigram_counts[previous_word]

    if denominator == 0:
        return 0

    return numerator / denominator


# =========================================================
# 5. Trigram Counts
# =========================================================

trigram_counts = Counter()

for sentence in corpus:
    for i in range(len(sentence) - 2):
        w1, w2, w3 = sentence[i:i + 3]
        trigram_counts[(w1, w2, w3)] += 1


def trigram_prob(w1, w2, w3):
    numerator = trigram_counts[(w1, w2, w3)]
    denominator = bigram_counts[(w1, w2)]

    if denominator == 0:
        return 0

    return numerator / denominator


# =========================================================
# 6. Generate Sentence using Trigram Model
# =========================================================

def sample_start_pair():
    """
    <s>로 시작하는 trigram에서
    첫 두 단어를 확률적으로 선택한다.
    """
    candidates = []
    weights = []

    for (w1, w2, w3), count in trigram_counts.items():
        if w1 == "<s>":
            candidates.append((w2, w3))
            weights.append(count)

    return random.choices(
        candidates,
        weights=weights,
        k=1
    )[0]


def sample_next_word_trigram(w1, w2):
    """
    이전 두 단어 (w1, w2)가 주어졌을 때
    다음 단어를 trigram count에 비례하여 선택한다.
    """
    candidates = []
    weights = []

    for (a, b, c), count in trigram_counts.items():
        if a == w1 and b == w2:
            candidates.append(c)
            weights.append(count)

    if not candidates:
        return "</s>"

    return random.choices(
        candidates,
        weights=weights,
        k=1
    )[0]


def generate_trigram_sentence(max_length=15):
    first, second = sample_start_pair()

    output = [first]

    if second == "</s>":
        return " ".join(output)

    output.append(second)

    w1 = first
    w2 = second

    while len(output) < max_length:
        next_word = sample_next_word_trigram(w1, w2)

        if next_word == "</s>":
            break

        output.append(next_word)

        w1, w2 = w2, next_word

    return " ".join(output)


# 결과 재현을 위한 seed
random.seed(42)

print("===== 1. Trigram Generated Sentences =====")

generated_sentences = []

while len(generated_sentences) < 5:
    sentence = generate_trigram_sentence()

    # 중복 없이 5문장 출력
    if sentence not in generated_sentences:
        generated_sentences.append(sentence)

for i, sentence in enumerate(generated_sentences, 1):
    print(f"{i}. {sentence}")

===== 1. Trigram Generated Sentences =====
1. can you find a chinese restaurant
2. tell me about chinese restaurants
3. i am looking for a restaurant
4. i want to eat
5. i want to eat thai food


Bigram 생성 방식은 이전 한 단어를 기준으로 다음 단어를 sampling하는 방식입니다. 여기서는 이를 Trigram으로 확장해서 **이전 두 단어 `(w1, w2)`를 보고 다음 단어 `w3`를 sampling**하도록 구현했습니다.

즉,

$$
P(w_3|w_1,w_2)
=
\frac{C(w_1,w_2,w_3)}
{C(w_1,w_2)}
$$

를 사용합니다.

참고로 생성된 문장이 학습 문장과 동일한 것은 오류가 아닙니다. 현재 corpus가 15문장으로 매우 작고 MLE만 사용하고 있기 때문에, 모델은 학습 때 관찰했던 trigram을 따라서 문장을 생성합니다.

---

**2. Unigram, Bigram, Trigram의 성능을 Perplexity metric을 사용하여 측정하고 비교하세요. 단, 테스트 sentence는 5개 이상으로 구성하되, 훈련에 사용된 sentence는 제외하세요.**

In [2]:
# =========================================================
# 7. Perplexity
# =========================================================

def unigram_perplexity(sentence):
    tokens = tokenize(sentence)

    # <s>는 context 역할이므로 제외하고 계산
    tokens = tokens[1:]

    log_prob = 0
    n = len(tokens)

    for word in tokens:
        p = unigram_prob(word)

        if p == 0:
            return float("inf")

        log_prob += math.log(p)

    return math.exp(-log_prob / n)


def bigram_perplexity(sentence):
    tokens = tokenize(sentence)

    log_prob = 0
    n = len(tokens) - 1

    for i in range(n):
        p = bigram_prob(
            tokens[i],
            tokens[i + 1]
        )

        if p == 0:
            return float("inf")

        log_prob += math.log(p)

    return math.exp(-log_prob / n)


def trigram_perplexity(sentence):
    tokens = tokenize(sentence)

    log_prob = 0
    n = len(tokens) - 2

    for i in range(n):
        p = trigram_prob(
            tokens[i],
            tokens[i + 1],
            tokens[i + 2]
        )

        if p == 0:
            return float("inf")

        log_prob += math.log(p)

    return math.exp(-log_prob / n)


# =========================================================
# 8. Test Sentences
#    - Training corpus에 없는 문장만 사용
# =========================================================

test_sentences = [
    "i want a chinese restaurant",
    "i want a thai restaurant",
    "i am looking for a chinese restaurant",
    "where can i eat thai food",
    "tell me about chinese food",
]

# training sentence가 test set에 들어가지 않았는지 확인
assert all(s not in sentences for s in test_sentences)


print("\n===== 2. Perplexity Comparison =====")

print(
    f"{'Sentence':45s}"
    f"{'Unigram':>12s}"
    f"{'Bigram':>12s}"
    f"{'Trigram':>12s}"
)

print("-" * 81)

for sentence in test_sentences:
    uni_pp = unigram_perplexity(sentence)
    bi_pp = bigram_perplexity(sentence)
    tri_pp = trigram_perplexity(sentence)

    print(
        f"{sentence:45s}"
        f"{uni_pp:12.4f}"
        f"{bi_pp:12.4f}"
        f"{tri_pp:12.4f}"
    )


# =========================================================
# 9. Corpus-level Perplexity
# =========================================================

def corpus_unigram_perplexity(sentences):
    log_prob = 0
    n = 0

    for sentence in sentences:
        tokens = tokenize(sentence)[1:]

        for word in tokens:
            p = unigram_prob(word)

            if p == 0:
                return float("inf")

            log_prob += math.log(p)
            n += 1

    return math.exp(-log_prob / n)


def corpus_bigram_perplexity(sentences):
    log_prob = 0
    n = 0

    for sentence in sentences:
        tokens = tokenize(sentence)

        for i in range(len(tokens) - 1):
            p = bigram_prob(tokens[i], tokens[i + 1])

            if p == 0:
                return float("inf")

            log_prob += math.log(p)
            n += 1

    return math.exp(-log_prob / n)


def corpus_trigram_perplexity(sentences):
    log_prob = 0
    n = 0

    for sentence in sentences:
        tokens = tokenize(sentence)

        for i in range(len(tokens) - 2):
            p = trigram_prob(
                tokens[i],
                tokens[i + 1],
                tokens[i + 2]
            )

            if p == 0:
                return float("inf")

            log_prob += math.log(p)
            n += 1

    return math.exp(-log_prob / n)


print("\n===== Corpus-level Perplexity =====")

print(
    "Unigram :",
    corpus_unigram_perplexity(test_sentences)
)

print(
    "Bigram  :",
    corpus_bigram_perplexity(test_sentences)
)

print(
    "Trigram :",
    corpus_trigram_perplexity(test_sentences)
)


===== 2. Perplexity Comparison =====
Sentence                                          Unigram      Bigram     Trigram
---------------------------------------------------------------------------------
i want a chinese restaurant                       15.2532      2.7144         inf
i want a thai restaurant                          16.3196      2.5370         inf
i am looking for a chinese restaurant             24.4697      2.1147         inf
where can i eat thai food                         22.0699      3.3565         inf
tell me about chinese food                        26.0827      1.6802         inf

===== Corpus-level Perplexity =====
Unigram : 20.64684468628891
Bigram  : 2.419595689453563
Trigram : inf


Perplexity는

$$
PP(W)
=
\exp
\left(
-\frac{1}{N}
\sum_i \log P(w_i|\text{context})
\right)
$$

으로 계산했으며, **값이 작을수록 모델이 해당 test data를 더 잘 예측했다는 의미**입니다. 
확률이 0인 n-gram을 만나면 Perplexity를 `inf`로 반환합니다.

---
**3. 정리**

Unigram, Bigram, Trigram Language Model을 MLE 방식으로 학습한 후, 학습 corpus에 포함되지 않은 5개의 문장을 대상으로 Perplexity를 측정하였습니다. 실험 결과 전체 test corpus의 Perplexity는 Unigram이 약 20.65, Bigram이 약 2.42였으며, Trigram은 infinity가 나타났습니다. 따라서 본 실험에서는 Bigram이 가장 낮은 Perplexity를 보였습니다.

Bigram은 바로 이전 단어를 조건으로 사용하므로 Unigram보다 문맥 정보를 활용할 수 있었고, 선택한 test sentence의 Bigram 조합들이 training corpus에서 관찰되었기 때문에 낮은 Perplexity를 보였습니다. 반면 Trigram은 이전 두 단어를 조건으로 사용하기 때문에 더 구체적인 문맥을 표현할 수 있지만, 작은 training corpus에서는 관찰되지 않은 Trigram이 쉽게 발생합니다. Test sentence에는 training corpus에서 관찰되지 않은 Trigram이 포함되어 있어 해당 확률이 0이 되었고, 이에 따라 Perplexity가 infinity가 되었습니다.

이 결과는 Trigram 자체의 성능이 항상 Bigram보다 낮다는 의미가 아니라, **학습 데이터가 15문장으로 매우 작은 상황에서 고차 N-gram 모델이 data sparsity와 zero-probability 문제에 더 민감하다는 것**을 보여줍니다. 더 많은 학습 데이터를 사용하거나 smoothing, backoff, interpolation 등의 방법을 적용하면 이러한 문제를 완화할 수 있습니다.

본 실험에서는 smoothing을 적용하지 않은 MLE 모델을 사용하였으므로, test sentence에 학습 과정에서 관찰되지 않은 n-gram이 하나라도 포함되면 해당 모델의 Perplexity가 infinity가 됩니다. 따라서 본 결과는 작은 학습 corpus와 선택된 test set에서의 비교 결과입니다.